# Brent · `valid_date` (web: "Valid date")

**Goal:** understand the data, so that a chat bot can answer any question about it
more easily and correctly.

Brent only (golden rule): every query carries the location-based Brent filter.
Data comes from the **live API**.

Workflow: add a step here → discuss the findings → decide whether it belongs in
`knowledge/brent.md` → add it.

In [1]:
from pathlib import Path

import httpx
import pandas as pd
from dotenv import dotenv_values

PROJECT_ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
env = dotenv_values(PROJECT_ROOT / ".env")

BASE_URL = "https://planningdata.london.gov.uk/api-guest"
HEADERS = {"X-API-AllowRequest": env["DATAHUB_ALLOW_REQUEST"]}


def brent_filter() -> dict:
    # Location-based scope: borough is Brent, or borough is missing and Brent decided it.
    return {
        "bool": {
            "should": [
                {"term": {"borough.raw": "Brent"}},
                {"bool": {
                    "filter": [{"term": {"lpa_name.raw": "Brent"}}],
                    "must_not": [{"exists": {"field": "borough.raw"}}],
                }},
            ],
            "minimum_should_match": 1,
        }
    }


def search(body: dict) -> dict:
    response = httpx.post(f"{BASE_URL}/applications/_search", headers=HEADERS, json=body, timeout=60)
    response.raise_for_status()
    return response.json()

## Step 1 · Nulls and values that are not dates

Fetch `valid_date` for every Brent application (paged with `search_after`, sorted by
`id`, because there are more than 10,000). Then count:
- **null**: no value at all;
- **empty or blank**: a string with nothing in it;
- **not a date**: a value that doesn't parse as `DD/MM/YYYY`.

In [2]:
rows, after = [], None
while True:
    body = {"size": 5000, "query": brent_filter(), "_source": ["id", "valid_date"], "sort": [{"id": "asc"}]}
    if after:
        body["search_after"] = after
    page = search(body)["hits"]["hits"]
    if not page:
        break
    rows += [h["_source"] for h in page]
    after = page[-1]["sort"]

df = pd.DataFrame(rows).reindex(columns=["id", "valid_date"])
print("Brent applications:", len(df))

Brent applications: 36952


In [3]:
value = df["valid_date"]
is_null = value.isna()
is_blank = value.notna() & (value.astype(str).str.strip() == "")
parsed = pd.to_datetime(value, format="%d/%m/%Y", errors="coerce")
not_a_date = value.notna() & ~is_blank & parsed.isna()

summary = pd.Series({
    "null": int(is_null.sum()),
    "empty or blank": int(is_blank.sum()),
    "not a date": int(not_a_date.sum()),
    "valid DD/MM/YYYY date": int(parsed.notna().sum()),
})
summary["total"] = summary.sum()
display(summary.to_frame("count").assign(**{"share_%": lambda t: (100 * t["count"] / len(df)).round(2)}))

print("Values that are not dates:")
df[not_a_date]

,count,share_%
null,3928,10.63
empty or blank,0,0.00
not a date,0,0.00
valid DD/MM/YYYY date,33024,89.37
total,36952,100.00


Values that are not dates:


,id,valid_date


### Findings · Step 1

Live API, 2026-11-09, 36,952 Brent applications:

| | Count | Share |
|---|---|---|
| null | 3,928 | 10.63% |
| empty or blank | 0 | 0% |
| not a date | 0 | 0% |
| valid `DD/MM/YYYY` date | 33,024 | 89.37% |

- **No weird values:** every `valid_date` that is filled in is a real `DD/MM/YYYY` date.
- **3,928 have no valid date** (10.6%). That's four times as many as `decision_date`
  has (995).

## Step 2 · Records with no valid date: do they have a decision date?

For the records with no `valid_date`, count how many have a `decision_date` and how many don't.
Uses `_count` with the Brent filter.

In [4]:
def count(*extra: dict) -> int:
    body = {"query": {"bool": {"filter": [brent_filter(), *extra]}}}
    response = httpx.post(f"{BASE_URL}/applications/_count", headers=HEADERS, json=body, timeout=60)
    response.raise_for_status()
    return response.json()["count"]


no_valid = {"bool": {"must_not": [{"exists": {"field": "valid_date"}}]}}
has_decision_date = {"exists": {"field": "decision_date"}}
no_decision_date = {"bool": {"must_not": [{"exists": {"field": "decision_date"}}]}}

pd.Series({
    "no valid_date": count(no_valid),
    "  with a decision_date": count(no_valid, has_decision_date),
    "  without a decision_date": count(no_valid, no_decision_date),
}).to_frame("count")

,count
no valid_date,3928
with a decision_date,3593
without a decision_date,335


### Findings · Step 2

Live API, 2026-11-09:

| No `valid_date` | Count |
|---|---|
| with a `decision_date` | 3,593 |
| without a `decision_date` | 335 |
| **total** | **3,928** |

- **Most (3,593, 91%) were decided** even though they have no valid date. So a missing
  valid date doesn't mean the application is new or pending; the date is just missing.
- **335 have neither date.**

## Step 3 · Overlap of the 335 with the 407

- **335**: no `valid_date` and no `decision_date` (Step 2).
- **407**: no `status`, no `decision` and no `decision_date` (found in the `decision` notebook).

In [5]:
def missing(field: str) -> dict:
    return {"bool": {"must_not": [{"exists": {"field": field}}]}}


the_335 = [missing("valid_date"), missing("decision_date")]
the_407 = [missing("status"), missing("decision"), missing("decision_date")]

pd.Series({
    "the 335": count(*the_335),
    "the 407": count(*the_407),
    "in both": count(missing("valid_date"), missing("status"), missing("decision"), missing("decision_date")),
}).to_frame("count")

,count
the 335,335
the 407,407
in both,322


### Findings · Step 3

Live API, 2026-11-09:

| Group | Count |
|---|---|
| in both | 322 |
| only in the 335 (they have a `status` or a `decision`) | 13 |
| only in the 407 (they have a `valid_date`) | 85 |

- **Mostly the same records:** 322 of the 335 (96%) are also in the 407. Those 322 have none
  of the 4 fields: no `valid_date`, `status`, `decision` or `decision_date`.

## Step 4 · Import source of the 3,928 with no valid date

Count the records with no `valid_date` by `bo_system` (how the record got into the Datahub).

In [6]:
body = {
    "size": 0,
    "query": {"bool": {"filter": [brent_filter(), missing("valid_date")]}},
    "aggs": {
        "bo_system": {"terms": {"field": "bo_system.raw", "size": 50}},
        "no_bo_system": {"missing": {"field": "bo_system.raw"}},
    },
}
aggs = search(body)["aggregations"]
by_source = pd.DataFrame(aggs["bo_system"]["buckets"]).rename(columns={"key": "bo_system", "doc_count": "count"})
by_source.loc[len(by_source)] = ["(null)", aggs["no_bo_system"]["doc_count"]]
by_source["share_%"] = (100 * by_source["count"] / by_source["count"].sum()).round(2)
print("total:", by_source["count"].sum())
by_source

total: 3928


,bo_system,count,share_%
0,FME Legacy Data Import,3531,89.89
1,CSV Data Import,338,8.60
2,Portal,59,1.50
3,(null),0,0.00


### Findings · Step 4

Live API, 2026-11-09, the 3,928 with no `valid_date`:

| `bo_system` | Count | Share |
|---|---|---|
| FME Legacy Data Import | 3,531 | 89.89% |
| CSV Data Import | 338 | 8.60% |
| Portal | 59 | 1.50% |

- **Most are legacy imports:** 3,531 (90%) came from `FME Legacy Data Import`.
- 397 are not legacy: 338 from `CSV Data Import` and 59 from `Portal`.

## Step 5 · All FME Legacy records in Brent

How many Brent records came from `FME Legacy Data Import`, and how many of them have a `valid_date`?

In [7]:
fme_legacy = {"term": {"bo_system.raw": "FME Legacy Data Import"}}

pd.Series({
    "FME Legacy records": count(fme_legacy),
    "  with a valid_date": count(fme_legacy, {"exists": {"field": "valid_date"}}),
    "  without a valid_date": count(fme_legacy, missing("valid_date")),
}).to_frame("count")

,count
FME Legacy records,3531
with a valid_date,0
without a valid_date,3531


### Findings · Step 5

Live API, 2026-11-09:

| FME Legacy records | Count |
|---|---|
| with a `valid_date` | 0 |
| without a `valid_date` | 3,531 |
| **total** | **3,531** |

- **No FME Legacy record has a valid date.** Legacy records never carry one, so for them a
  missing `valid_date` is not a gap in individual records.
- So the 3,928 split into 3,531 legacy records (always missing) and 397
  non-legacy records (338 CSV import, 59 Portal) that are each missing one.

## Step 6 · The 397 non-legacy gaps without the empty records

The 407 empty records (no `status`, `decision`, `decision_date` or `description`) are
dropped at load. How many of the 397 non-legacy records with no `valid_date` are left?

In [8]:
empty_record = {"bool": {"filter": [missing("status"), missing("decision"), missing("decision_date"), missing("description")]}}


def count_not(filters: list[dict], not_filters: list[dict]) -> int:
    body = {"query": {"bool": {"filter": [brent_filter(), *filters], "must_not": not_filters}}}
    response = httpx.post(f"{BASE_URL}/applications/_count", headers=HEADERS, json=body, timeout=60)
    response.raise_for_status()
    return response.json()["count"]


pd.Series({
    "no valid_date": count(missing("valid_date")),
    "  FME Legacy": count(missing("valid_date"), fme_legacy),
    "  empty records (dropped)": count(missing("valid_date"), empty_record),
    "  empty records that are FME Legacy": count(missing("valid_date"), empty_record, fme_legacy),
    "  neither (real gaps)": count_not([missing("valid_date")], [fme_legacy, empty_record]),
}).to_frame("count")

,count
no valid_date,3928
FME Legacy,3531
empty records (dropped),322
empty records that are FME Legacy,0
neither (real gaps),75


### Findings · Step 6

Live API, 2026-11-09, the 3,928 with no `valid_date`:

| Group | Count |
|---|---|
| FME Legacy records (never have a valid date) | 3,531 |
| empty records (dropped at load) | 322 |
| real gaps | 75 |

- No empty record is an FME Legacy record, so the groups don't overlap.
- **After cleaning, only 75 real records are missing a valid date** that should have one.